# Activation bottleneck on TinyStories — Google Colab

Four matched runs: a standard transformer, the same model with a **hard TopK
activation bottleneck** in front of every MLP, and that bottleneck trained with
the **adaptive LapSum Top-(K+J) surrogate gradient**.

This is the *activation*-sparsity experiment and is unrelated to the repo's
weight-sparsity methods (`ltp`, `cs`, `topk`); enabling both at once is a config
error. `W_in` and `W_out` here are ordinary dense layers trained normally.

**Runtime → Change runtime type → GPU.** An A100/L4 is comfortable; a T4 works
if you drop `micro_batch_size`.

Data and checkpoints live on Drive under `MyDrive/weight-sparsity/` so they
survive a disconnect.

In [ ]:
!nvidia-smi

## 1. The three setups

| run | config | forward through the MLP input | backward |
| --- | --- | --- | --- |
| **dense** | `configs/bn_dense.yaml` | the ordinary transformer | ordinary |
| **hard** | `configs/bn_hard.yaml` | `W_in` → hard TopK (K of N) → `W_out` | hard mask only; the J candidates get nothing |
| **lapsum** | `configs/bn_lapsum.yaml` | **identical to `hard`** | LapSum Top-(K+J) surrogate, constant temperature `T` |

The comparison is built to isolate one thing at a time:

* **dense vs hard** — what the activation bottleneck itself costs.
* **hard vs lapsum** — what the surrogate gradient buys. These two share an
  architecture, a parameter count and a *bit-identical forward pass*; they
  differ only in what flows backwards through the TopK boundary.

The rank-boundary variant is the same run with
`--activation_bottleneck.surrogate_mode=rblapsum` on top of `bn_lapsum`: same
forward, and the boundary is the hard rank `max(b0, s_(K+1))` instead of the
soft budget `Σp = K`. `rblapsum_sf` puts the probabilities in the forward
(`y_i = z_i p_i`).

Sizes: the dense baseline is **81.7M** parameters (10 layers, `d_model=640`,
GPT-Neo vocab). The bottleneck adds `2 · d_model · N` per bottlenecked layer —
**26.2M** at `N=2048` across all 10 layers, so the bottlenecked runs are 107.9M.
That gap is unavoidable (the projections are real parameters) and is why
`hard` — not `dense` — is the control for the surrogate.


## 2. Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# --- Google Drive paths (persist across sessions) ---------------------------
PROJECT     = '/content/drive/MyDrive/weight-sparsity'
DRIVE_DATA  = f'{PROJECT}/data/tinystories'   # cached token stream
RUNS_DIR    = f'{PROJECT}/runs_bottleneck'    # checkpoints + metrics

# --- local scratch (fast, wiped when the runtime dies) ----------------------
REPO_DIR    = '/content/weight-sparsity'
LOCAL_DATA  = '/content/data/tinystories'     # what training actually reads

for p in (PROJECT, DRIVE_DATA, RUNS_DIR, LOCAL_DATA):
    os.makedirs(p, exist_ok=True)

print('drive project :', PROJECT)
print('runs          :', RUNS_DIR)

## 3. Clone the repo and install

In [ ]:
REPO_URL = 'https://github.com/labofdoubt/weight-sparsity.git'

if not os.path.exists(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull -q

%cd $REPO_DIR
!git log --oneline -1

In [ ]:
# Colab already ships torch, so install the package without pulling it again.
!pip install -q datasets transformers tokenizers pyyaml tqdm tensorboard
!pip install -q -e . --no-deps

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 4. Prepare the data (once)

Downloads TinyStories, tokenises it with the GPT-Neo tokenizer and writes flat
`uint16` streams. All three setups read the same stream.

In [ ]:
TOKENIZER = 'gpt_neo'   # or 'bpe'

drive_ready = os.path.exists(f'{DRIVE_DATA}/meta.json')

if drive_ready:
    print('copying the cached dataset from Drive ...')
    !cp -u $DRIVE_DATA/*.bin $DRIVE_DATA/meta.json $LOCAL_DATA/
else:
    print('preparing the dataset from scratch ...')
    cmd = (f'python -m wsparse.data --config configs/bn_dense.yaml '
           f'--data.data_dir={LOCAL_DATA} --data.tokenizer={TOKENIZER} '
           f'--data.tokenizer_path={LOCAL_DATA}/tokenizer --data.num_proc=2')
    !{cmd}
    print('caching it to Drive for next time ...')
    !cp -u $LOCAL_DATA/*.bin $LOCAL_DATA/meta.json $DRIVE_DATA/
    !cp -ru $LOCAL_DATA/tokenizer $DRIVE_DATA/ 2>/dev/null || true

!ls -lh $LOCAL_DATA
!cat $LOCAL_DATA/meta.json

## 5. Pick a setup and the shared hyper-parameters

Run this notebook three times, once per `SETUP`, or set `SETUP` and re-run the
train cell. The overrides below apply to all three so the comparison stays fair;
the bottleneck knobs are ignored by `dense`.

In [ ]:
SETUP = 'lapsum'          # 'dense' | 'hard' | 'lapsum'

CONFIGS = {
    'dense':  'configs/bn_dense.yaml',
    'hard':   'configs/bn_hard.yaml',
    'lapsum': 'configs/bn_lapsum.yaml',
}
RUN_NAMES = {name: f'bn_{name}' for name in CONFIGS}

CONFIG   = CONFIGS[SETUP]
RUN_NAME = RUN_NAMES[SETUP]

OVERRIDES = [
    f'--data.data_dir={LOCAL_DATA}',
    f'--train.out_dir={RUNS_DIR}',
    f'--train.run_name={RUN_NAME}',
    # --- shared across all three, so the comparison is fair ----------------
    '--train.max_steps=20000',
    '--train.batch_size=64',
    '--train.micro_batch_size=16',   # lower this if you hit OOM
    '--train.lr=6e-4',
    '--train.warmup_steps=500',
    '--train.dtype=bfloat16',
    '--train.compile=false',        # the gate graph-breaks; see the note below
    '--train.validate_every_steps=500',
    '--train.checkpoint_every_steps=2000',
    '--train.sample_every_steps=5000',
    '--train.seed=1337',            # same init for all three
    # --- bottleneck geometry (ignored by SETUP='dense') -------------------
    '--activation_bottleneck.n_features=2048',   # N
    '--activation_bottleneck.k=256',             # K active: 12.5% of N, 0.40x d_model
    '--activation_bottleneck.j=768',             # J extra candidates that only get gradient
    '--activation_bottleneck.layers=all',        # all | even | odd | first:n | last:n | 0,2,4
    '--activation_bottleneck.selection_mode=abs_topk',   # topk | abs_topk | gated_topk
    # --- the surrogate (the only thing that differs between hard and lapsum) -
    # One constant temperature, in score units, shared by lapsum and rblapsum.
    '--activation_bottleneck.temperature=1.0',
    '--activation_bottleneck.log_diagnostics=true',
    # --- rblapsum family (only read by surrogate_mode=rblapsum[_sf]) --------
    # Same hard forward; the boundary is the hard rank b = max(b0, s_(K+1))
    # instead of the soft budget sum p_i = K.
    # '--activation_bottleneck.surrogate_mode=rblapsum',
    # '--activation_bottleneck.rblapsum_boundary_grad_mode=through_rank_kappa',
    #                                  # detach | project | through_rank | through_rank_kappa
    # '--activation_bottleneck.rblapsum_boundary_floor=0.0',   # b0
    # '--activation_bottleneck.rblapsum_support_scale=1.0',    # scales g_s only
]
OVERRIDE_STR = ' '.join(OVERRIDES)
print(SETUP, '->', CONFIG)
print(OVERRIDE_STR)

`train.compile=false`: the gate contains a custom autograd Function and
data-dependent solver control flow, so `torch.compile` graph-breaks around it.
The dense baseline can be compiled safely — but leave it off for all three if
you want the wall-clock numbers to be comparable.

### Parameter counts for all three

In [ ]:
for name, cfg in CONFIGS.items():
    print('=' * 78)
    !python scripts/model_summary.py --config $cfg $OVERRIDE_STR

## 6. Train

The log line adds `t` (adaptive temperature), `t/std` (relative to the score
scale), `neff` (realized effective count) and `dK` (`|Σp − K|`, the barrier
residual) for the bottlenecked runs.

**Watch `neff`**: if it drifts off its target, or `bottleneck/status_*` shows
rows flagged, the target is not attainable for that score geometry.

In [ ]:
!python -m wsparse.train --config $CONFIG $OVERRIDE_STR

Re-run with `--train.resume=auto` appended after a disconnect. To get all three
runs, set `SETUP` above to each of `dense`, `hard`, `lapsum` and re-run the two
cells.

In [ ]:
# !python -m wsparse.train --config $CONFIG $OVERRIDE_STR --train.resume=auto

## 7. TensorBoard

Every run writes events to `<RUNS_DIR>/<run_name>/tb`, so pointing TensorBoard
at `RUNS_DIR` overlays all the setups in one chart. Metric names are already
namespaced, so the panels group themselves:

| section | what to watch |
| --- | --- |
| `train/`, `val/` | `ce`, `ppl`, `lr`, `grad_norm` |
| `perf/` | `tokens_per_s`, `ms_per_step` — the surrogate's real cost |
| `bottleneck/temperature` | the constant `T` the kernel/barrier runs at (flat by construction) |
| `bottleneck/barrier`, `barrier_gap`, `frac_above_barrier` | where the LapSum barrier sits relative to `s_(K+1)` |
| `bottleneck/budget_residual`, `barrier_failures` | `\|Σp − K\|`; should sit at ~1e-6, a rise means the barrier solve is struggling |
| `bottleneck/score_gap`, `score_span` | the score geometry the boundary lives in |
| `bottleneck/feature_*` | **dead features and usage collapse — see below** |
| `bottleneck/grad_active`, `grad_inactive`, `grad_by_rank` | surrogate gradient magnitude on the active K, the exploratory J, and by candidate rank |
| `bottleneck/rb_*` (rblapsum only) | boundary rank, cap-active fraction, support-gradient norm, common mode |

`bottleneck/feature_dead_frac` and `feature_usage_entropy` are the ones to keep
an eye on. The characteristic failure of a TopK bottleneck is collapse: a subset
of the N features wins every token, the rest are never selected, and their
`W_in`/`W_out` columns stop receiving gradient entirely — so the effective width
is far below N. The loss and the budget both look perfectly healthy while that
happens. `feature_usage_entropy` is `exp(H)/N`: 1.0 is even usage, and it falls
towards the surviving fraction as features die.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir $RUNS_DIR

## 8. Curves

In [ ]:
import json, os
import matplotlib.pyplot as plt

def load(name):
    path = f'{RUNS_DIR}/{RUN_NAMES[name]}/metrics.jsonl'
    if not os.path.exists(path):
        return []
    return [json.loads(l) for l in open(path)]

def series(records, key):
    xs = [(r['step'], r[key]) for r in records if key in r]
    return [x for x, _ in xs], [y for _, y in xs]

runs = {name: load(name) for name in CONFIGS}
runs = {k: v for k, v in runs.items() if v}
colors = {'dense': 'tab:grey', 'hard': 'tab:orange', 'lapsum': 'tab:blue'}

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))

for name, rec in runs.items():
    s, v = series(rec, 'train/ce')
    axes[0].plot(s, v, color=colors[name], alpha=.55, lw=1)
    s, v = series(rec, 'val/ce')
    axes[0].plot(s, v, 'o-', color=colors[name], label=name, ms=3)
axes[0].set_xlabel('step'); axes[0].set_ylabel('cross-entropy')
axes[0].set_title('loss (faint = train, dots = val)'); axes[0].legend(); axes[0].grid(alpha=.3)

# what the surrogate is doing: how much gradient reaches the J candidates that
# the forward pass dropped.  Exactly 0 for hard, by construction.
for name, rec in runs.items():
    s, v = series(rec, 'bottleneck/grad_inactive')
    if s:
        axes[1].plot(s, v, color=colors[name], label=f'{name}: J')
    s, v = series(rec, 'bottleneck/grad_active')
    if s:
        axes[1].plot(s, v, '--', color=colors[name], alpha=.7, label=f'{name}: K')
axes[1].set_xlabel('step'); axes[1].set_ylabel('mean |dL/ds|')
axes[1].set_yscale('log')
axes[1].set_title('surrogate gradient (dashed = active K)')
axes[1].legend(); axes[1].grid(alpha=.3)

# is the bottleneck using its width, or collapsing onto a few features?
for name, rec in runs.items():
    s, v = series(rec, 'bottleneck/feature_usage_entropy')
    if s:
        axes[2].plot(s, v, color=colors[name], label=f'{name}: exp(H)/N')
    s, v = series(rec, 'bottleneck/feature_dead_frac')
    if s:
        axes[2].plot(s, v, ':', color=colors[name], alpha=.8, label=f'{name}: dead')
axes[2].set_ylim(0, 1.02)
axes[2].set_xlabel('step'); axes[2].set_ylabel('fraction of N')
axes[2].set_title('feature usage (dotted = dead fraction)')
axes[2].legend(); axes[2].grid(alpha=.3)

plt.tight_layout(); plt.show()


## 9. Compare the setups

In [ ]:
import math

rows = []
for name, rec in runs.items():
    val = [r for r in rec if 'val/ce' in r]
    # filter on density, not temperature: the hard baseline runs no solver but
    # still reports the forward-side statistics
    last = [r for r in rec if 'bottleneck/density' in r]
    best = min((r['val/ce'] for r in val), default=float('nan'))
    row = {'setup': name,
           'best val ce': round(best, 4),
           'val ppl': round(math.exp(best), 2) if best == best else float('nan')}
    if last:
        d = last[-1]
        nan = float('nan')
        row.update({
            'K/N': round(d['bottleneck/density'], 4),
            'T': round(d.get('bottleneck/temperature', nan), 4),
            '|sum p - K|': f"{d['bottleneck/budget_residual']:.1e}" if 'bottleneck/budget_residual' in d else '-',
            'grad on J': f"{d.get('bottleneck/grad_inactive', 0.0):.2e}",
            'rows r_K+1 > b': round(d.get('bottleneck/frac_above_barrier', nan), 3),
            'dead features': round(d.get('bottleneck/feature_dead_frac', nan), 3),
        })
    rows.append(row)

try:
    import pandas as pd
    display(pd.DataFrame(rows).set_index('setup'))
except ImportError:
    for r in rows:
        print(r)

print()
print('dense vs hard   : what the activation bottleneck itself costs.')
print('hard  vs lapsum : what the surrogate buys.  Same architecture, same')
print('                  parameter count, bit-identical forward pass -- the only')
print('                  difference is the backward.')
print()
print('grad on J is the tell: exactly 0.00e+00 for hard (the J inactive candidates')
print('receive nothing), non-zero for lapsum (they receive the boundary-exchange')
print('gradient).  NaNs in the T and barrier columns are expected for dense and')
print('hard -- neither solves a barrier.')


## 10. Sample from a checkpoint

In [ ]:
CKPT = f'{RUNS_DIR}/{RUN_NAME}/latest.pt'
!python scripts/generate.py --ckpt "$CKPT" --prompt "Once upon a time" --tokens 200

## 11. Notes

Checkpoints and `metrics.jsonl` are already on Drive under
`MyDrive/weight-sparsity/runs_bottleneck/`, so nothing else is needed to keep
the results.

Knobs worth sweeping, in rough order of interest:

* `k` — the FLOP budget. `k=256` of `N=2048` is 12.5%, and 0.40× `d_model`.
* `j` — how many inactive candidates get gradient. `j=0` is not allowed; set
  `surrogate_mode=hard` for no surrogate at all.
* `temperature` — the kernel/barrier bandwidth, in score units. Shared by the
  lapsum and rblapsum modes; larger spreads the boundary gradient wider.
* `surrogate_mode` — `lapsum` (soft budget) vs `rblapsum` (hard rank boundary)
  vs `rblapsum_sf` (probabilities in the forward).
* `layers` — `even` halves the added parameters.
* `post_norm` — an RMSNorm on each bottleneck's output; matters under the
  stream placements, where the output scale compounds with depth.
